In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('work with null').getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()
print(uploaded)

Saving bookings.csv to bookings.csv
{'bookings.csv': b'booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode\r\n1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI\r\n1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card\r\n1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI\r\n1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash\r\n1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card\r\n1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,\r\n1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI\r\n1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card\r\n1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI\r\n1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card\r\n1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,\r\n1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI\r\n1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash\r\n1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card\r\n1015

In [6]:
bookings_df = spark.read.csv('bookings.csv', header=True, inferSchema=True)
bookings_df.printSchema()
bookings_df.show()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|   

In [7]:
from pyspark.sql.functions import max, min, sum, avg, col, array_contains, when

bookings_df.filter(col('city').isNull()).show()
bookings_df.filter(col('provider').isNull()).show()

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|         UPI|
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hyderabad|       Hotel|    NULL|          7500|  

In [8]:
drop_any_null_df = bookings_df.na.drop()
drop_any_null_df.show()

clean_df = bookings_df.na.drop(
    subset = [
        'customer_name',
        'service_type',
        'booking_amount'
    ]
)
clean_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1008|   Meera Nair|    Kochi|       Hotel|Hill View Resort|          7500|     Confirmed|        Card|
|      1009|    Rohan Das|  Kolkata|      Flight|       Air India|          7400|       Pending|         UPI|
|      1010|  Nisha Reddy|Bangalore|      Flight| British Airways|         62000|     Confirmed|        Card|
|      1015|   Ravi Kumar|    Delhi|      Flight|        SpiceJet|          4800|     Confirmed|         UPI|
+---------

In [9]:
filled_df = bookings_df.na.fill({
    'city': 'Unknown'
})
filled_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [11]:
filled_df = bookings_df.na.fill({
    'city': 'Unknown',
    'provider': 'Not Available',
    'booking_status': 'Unknown',
    'payment_mode': 'Not Provided',
    'booking_amount': 0
})
filled_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|Not Provided|
|      100